# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [6]:
# Only needed for Udacity workspace

import importlib.util
import sys
import json

from pydantic import BaseModel

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [7]:

import os
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from tavily import TavilyClient

In [8]:

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [9]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

In [10]:
class EvaluationReport(BaseModel):
    grounded: bool
    has_sufficient_context: bool
    explanation: str

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [11]:

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")


#### Evaluate Retrieval Tool

In [13]:
@tool
def retrieve_game(query: str) -> str:
    results = collection.query(
        query_texts = [query],
        n_results =3
    )

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]

    output = []

    for i, doc in enumerate(documents):
        meta = metadatas[i] if i < len(metadatas) else {}
        output.append(
            f"Result {i+1}: \n"
            f"Title: {meta.get('title', 'Unknown')} \n"
            f"Genre: {meta.get('genre', 'Unknown')}\n"
            f"Platform: {meta.get('platform', 'Unknown')}\n"
            f"Content: {doc}"
        )

        return "\n\n".join(output)




In [14]:
@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> dict:
    retrieved_docs = retrieved_docs[:4000]
    
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    evaluator = LLM(model="gpt-4o-mini")

    prompt = f""" 

    Your task is to evaluate if the documents are enough to respond to the query.
    Give a detailed explanation, so it's possible to take an action to accept it or not.

    User question:
    {question}

    Retrieved documents:
    {retrieved_docs}

    Decide:
    1. Are the retrieved documents relevant to the question?
    2. Do they contain enough information to answer it confidently?
    3. Is important information missing?

    Return:
    - useful: true or false
    - description: a short explanation of your decision
    """

    result = evaluator.invoke(
        input=[UserMessage(content=prompt)],
        response_format = EvaluationReport
    )

    report = result.content

    if isinstance(report, str):
        report = json.loads(report)

    return {
        "grounded": report["grounded"],
        "has_sufficient_context": report["has_sufficient_context"],
        "explanation": report["explanation"],
    }

#### Game Web Search Tool

In [15]:
@tool
def game_web_serach(question: str) -> str:

    """
    Semantic search: Finds most results in the vector DB

    args: 
      - question: a question about game industry.
    """

    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=5
    )

    return str(response)

### Agent

In [16]:
agent = Agent(
    model_name = "gpt-4o-mini" ,
    instructions="""
    You are an AI research assistant for the video game industry.

    Use the available tools when needed:
    - First, use retrieve_game to search the internal game database.
    - Then, use evaluate_retrieval to decide whether the retrieved documents are enough.
    - If the retrieved documents are not enough, use game_web_search to gather more current or missing information.
    - Provide clear, concise, and helpful answers.
    - If the information is uncertain or incomplete, say so.
    """,
    tools=[retrieve_game, evaluate_retrieval, game_web_serach],
    temperature=0.0

)

In [17]:
questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?"
]

for question in questions:
    response = agent.invoke(question)
    print(f"Question: {question}")
    print(f"Answer: {response}")
    print("-" * 50)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Question: When Pokémon Gold and Silver was released?
Answer: Run('d93abef5-7280-49ef-bba1-37c972e15817')
--------------------------------------------------
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Question: Which one was the first 3D platformer Mario game?
Answer: Run('5a9a3d18-c6c3-4c1e-9eef-33ba79eeff8e')
------------------------------------

### (Optional) Advanced

In [ ]:
agent = Agent(
    model_name="gpt-4o-mini",
    instructions="You are a helpful game research assistant.",
    tools=[retrieve_game, evaluate_retrieval, game_web_serach]  # include your tools here
)
run = agent.invoke("What are the recent trends in open-world survival games?")
final_state = run.get_final_state()
print(final_state["messages"][-1].content)